# Pipeline de vídeo optimizado

Mismo resultado que `baseline_CV_reto_VV.ipynb` (secciones 1, 6 y 7) + `active_speaker_recognition.ipynb`, reorganizado para el backend:

- **Una sola fuente de fotogramas** (25 fps, extraídos una vez) para todos los pasos: sin `cap.set` ni desfases.
- **Una sola pasada de detección**: cada cara se detecta una vez y la reutilizan TalkNet (todos los fotogramas), MiVOLO y ArcFace (1 de cada `STRIDE`).
- **TransNetV2** para los cortes de plano (detecta también fundidos, que PySceneDetect no ve).
- **YOLO-pose + BoT-SORT con ReID**: menos cambios de identidad dentro de un plano; la pose filtra manos y brazos.
- **ArcFace** sigue haciendo falta: une tracks de la misma persona *entre* planos, algo que el tracker no puede hacer.

## 0. Configuración y modelos

In [ ]:
import os, sys, glob, math, shutil, subprocess
import cv2, numpy as np, pandas as pd, torch, tqdm, python_speech_features, gdown
import ultralytics, transnetv2_pytorch
from collections import Counter
from scipy import signal
from scipy.io import wavfile
from scipy.interpolate import interp1d
from scipy.cluster.hierarchy import linkage, fcluster
from ultralytics import YOLO
from insightface.app import FaceAnalysis
from transformers import AutoModelForImageClassification, AutoConfig, AutoImageProcessor
from transnetv2_pytorch import TransNetV2
from IPython.display import Video, display

VIDEO = "videos/robot_social_CSIC.mp4"
FPS = 25              # todo el pipeline trabaja a 25 fps, lo que necesita TalkNet
STRIDE = 4            # personas, MiVOLO y ArcFace en 1 de cada 4 fotogramas; caras para TalkNet en todos
MIN_SECONDS = 1.0     # duración mínima de un track
MIN_HEAD = 0.5        # fracción mínima de detecciones con cabeza u hombros visibles (descarta manos y brazos)
MAX_DIST = 0.6        # reidentificación: distancia del coseno máxima entre tracks de la misma persona
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
PROVIDERS = ["CUDAExecutionProvider", "CPUExecutionProvider"]

work = os.path.join("pipeline_work", os.path.splitext(os.path.basename(VIDEO))[0])
os.makedirs(os.path.join(work, "frames"), exist_ok=True)

# BoT-SORT con ReID: copia del YAML de Ultralytics con with_reid activado ("auto" = características del propio YOLO)
TRACKER = os.path.join(work, "botsort_reid.yaml")
with open(os.path.join(os.path.dirname(ultralytics.__file__), "cfg/trackers/botsort.yaml")) as src, open(TRACKER, "w") as dst:
    dst.write(src.read().replace("with_reid: False", "with_reid: True"))

pose = YOLO("yolo11m-pose.pt")
app_det = FaceAnalysis(name="buffalo_l", allowed_modules=["detection"], providers=PROVIDERS)                 # todos los fotogramas
app_id = FaceAnalysis(name="buffalo_l", allowed_modules=["detection", "recognition"], providers=PROVIDERS)   # + ArcFace
for a in (app_det, app_id):
    a.prepare(ctx_id=0, det_size=(640, 640))

mv_config = AutoConfig.from_pretrained("iitolstykh/mivolo_v2", trust_remote_code=True)
mivolo = AutoModelForImageClassification.from_pretrained("iitolstykh/mivolo_v2", trust_remote_code=True,
                                                         torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32).to(DEVICE)
mv_proc = AutoImageProcessor.from_pretrained("iitolstykh/mivolo_v2", trust_remote_code=True)

transnet = TransNetV2(device=DEVICE).eval()
transnet.load_state_dict(torch.load(os.path.join(os.path.dirname(transnetv2_pytorch.__file__), "transnetv2-pytorch-weights.pth"),
                                    map_location=transnet.device))

if not os.path.isdir("TalkNet-ASD"):
    subprocess.run("git clone -q --depth 1 https://github.com/TaoRuijie/TalkNet-ASD.git", shell=True, check=True)
pretrainModel = "TalkNet-ASD/pretrain_TalkSet.model"
if not os.path.isfile(pretrainModel):
    gdown.download(id="1AbN9fCf9IexMxEKXLQY2KYBlb-IhSEea", output=pretrainModel)
sys.path.insert(0, "TalkNet-ASD")
from talkNet import talkNet

## 1. Fotogramas a 25 fps y cortes de plano (TransNetV2)

In [ ]:
video25, audioFilePath = os.path.join(work, "video.avi"), os.path.join(work, "audio.wav")
subprocess.call(f"ffmpeg -y -i {VIDEO} -qscale:v 2 -async 1 -r 25 {video25} -loglevel panic", shell=True)
subprocess.call(f"ffmpeg -y -i {video25} -qscale:a 0 -ac 1 -vn -ar 16000 {audioFilePath} -loglevel panic", shell=True)
subprocess.call(f"ffmpeg -y -i {video25} -qscale:v 2 -f image2 {work}/frames/%06d.jpg -loglevel panic", shell=True)
flist = sorted(glob.glob(f"{work}/frames/*.jpg"))

with torch.no_grad():
    cuts = [int(s["start_frame"]) for s in transnet.detect_scenes(video25)[1:]]
shot_of = lambda f: int(np.searchsorted(cuts, f, side="right"))
print(len(flist), "fotogramas,", len(cuts) + 1, "planos")

## 2. Una sola pasada de detección

Todos los fotogramas: caras (para TalkNet). Fotogramas analizados: además personas con pose (YOLO + BoT-SORT), ArcFace y MiVOLO (género y edad).

In [ ]:
def crop(img, box):
    x1, y1, x2, y2 = [max(int(v), 0) for v in box]
    c = img[y1:y2, x1:x2]
    return c if c.size else None

def face_in_person(face_box, person_boxes):
    """Persona más estrecha cuya caja contiene el centro de la cara (regla del baseline), o None."""
    cx, cy = (face_box[0] + face_box[2]) / 2, (face_box[1] + face_box[3]) / 2
    inside = [i for i, b in enumerate(person_boxes) if b[0] <= cx <= b[2] and b[1] <= cy <= b[3]]
    return min(inside, key=lambda i: person_boxes[i][2] - person_boxes[i][0]) if inside else None

faces, persons = [], []          # faces: por fotograma (formato de TalkNet); persons: una fila por persona y fotograma analizado
for f, fname in enumerate(tqdm.tqdm(flist)):
    img = cv2.imread(fname)
    if f % STRIDE != STRIDE - 1:
        faces.append([{"frame": f, "bbox": fc.bbox.tolist()} for fc in app_det.get(img)])
        continue
    fcs = app_id.get(img)
    faces.append([{"frame": f, "bbox": fc.bbox.tolist()} for fc in fcs])
    r = pose.track(img, persist=True, tracker=TRACKER, verbose=False)[0]
    if r.boxes.id is None:
        continue
    boxes = r.boxes.xyxy.tolist()
    face_of = {}
    for fc in fcs:
        if (p := face_in_person(fc.bbox, boxes)) is not None:
            face_of[p] = fc
    with torch.no_grad():
        out = mivolo(faces_input=mv_proc(images=[crop(img, face_of[i].bbox) if i in face_of else None for i in range(len(boxes))])["pixel_values"].to(DEVICE, mivolo.dtype),
                     body_input=mv_proc(images=[crop(img, b) for b in boxes])["pixel_values"].to(DEVICE, mivolo.dtype))
    head = (r.keypoints.conf[:, :7] > 0.5).any(dim=1).tolist()        # nariz, ojos, orejas u hombros
    for i, (b, tid) in enumerate(zip(boxes, r.boxes.id.int().tolist())):
        persons.append({"frame": f, "t": f / FPS, "track": tid, "x1": b[0], "y1": b[1], "x2": b[2], "y2": b[3], "head": head[i],
                        "gender": "F" if mv_config.gender_id2label[int(out.gender_class_idx[i, 0])] == "female" else "M",
                        "age": float(out.age_output[i, 0]), "emb": face_of[i].normed_embedding if i in face_of else None})
df = pd.DataFrame(persons)
print(len(df), "detecciones de personas")

## 3. Tracks: un track por plano, filtro por pose, género y edad

In [ ]:
def vote(v, min_n=2, min_agree=0.8):
    if len(v) < min_n: return "unknown"
    label, n = Counter(v).most_common(1)[0]
    return label if n / len(v) >= min_agree else "unknown"

df["shot"] = df.frame.apply(shot_of)
df["track"] = df.groupby(["track", "shot"]).ngroup() + 1               # un track nuevo en cada plano
stats = df.groupby("track").agg(seconds=("frame", lambda f: len(f) * STRIDE / FPS), head=("head", "mean"))
keep = stats.index[(stats.seconds >= MIN_SECONDS) & (stats["head"] >= MIN_HEAD)]
print(f"Descartamos {len(stats) - len(keep)} de {len(stats)} tracks (cortos o sin cabeza ni hombros)")
df = df[df.track.isin(keep)].copy()

per_track = df.groupby("track").agg(shot=("shot", "first"), start_s=("t", "min"), end_s=("t", "max"),
                                    screen_time_s=("frame", lambda f: len(f) * STRIDE / FPS),
                                    gender=("gender", vote), age=("age", "median"))

## 4. Re-identificación entre planos (ArcFace)

In [ ]:
con_cara = df[df.emb.notna()]
track_emb = {t: np.mean(np.stack(g.emb), axis=0) for t, g in con_cara.groupby("track")}
track_emb = {t: e / np.linalg.norm(e) for t, e in track_emb.items()}
with_face = list(track_emb)
groups = [1] * len(with_face)
if len(with_face) > 1:
    groups = fcluster(linkage(np.stack([track_emb[t] for t in with_face]), method="average", metric="cosine"),
                      t=MAX_DIST, criterion="distance").tolist()
track_person = dict(zip(with_face, groups))
next_id = max(groups, default=0) + 1
for t in per_track.index:                                             # tracks sin cara: una persona cada uno
    if t not in track_person:
        track_person[t] = next_id; next_id += 1

df["person"] = df.track.map(track_person)
per_track["person"] = per_track.index.map(track_person)
per_track["gender_person"] = per_track.person.map(df.groupby("person").gender.agg(vote))
per_track["age_person"] = per_track.person.map(df.groupby("person").age.median().round())
print(len(per_track), "tracks ->", per_track.person.nunique(), "personas")

## 5. ¿Quién habla? (TalkNet sobre las caras ya detectadas)

In [ ]:
# Funciones copiadas de demoTalkNet.py (https://github.com/TaoRuijie/TalkNet-ASD/blob/main/demoTalkNet.py)
import numpy                                       # el código de TalkNet usa "numpy", no "np"
minTrack, numFailedDet, minFaceSize, cropScale, nDataLoaderThread = 10, 10, 1, 0.40, 10
def bb_intersection_over_union(boxA, boxB):
	xA = max(boxA[0], boxB[0])
	yA = max(boxA[1], boxB[1])
	xB = min(boxA[2], boxB[2])
	yB = min(boxA[3], boxB[3])
	interArea = max(0, xB - xA) * max(0, yB - yA)
	boxAArea = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
	boxBArea = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])
	return interArea / float(boxAArea + boxBArea - interArea)

def track_shot(sceneFaces):
	iouThres  = 0.5     # Minimum IOU between consecutive face detections
	tracks    = []
	while True:
		track     = []
		for frameFaces in sceneFaces:
			for face in frameFaces:
				if track == []:
					track.append(face)
					frameFaces.remove(face)
				elif face['frame'] - track[-1]['frame'] <= numFailedDet:
					iou = bb_intersection_over_union(face['bbox'], track[-1]['bbox'])
					if iou > iouThres:
						track.append(face)
						frameFaces.remove(face)
						continue
				else:
					break
		if track == []:
			break
		elif len(track) > minTrack:
			frameNum    = numpy.array([ f['frame'] for f in track ])
			bboxes      = numpy.array([numpy.array(f['bbox']) for f in track])
			frameI      = numpy.arange(frameNum[0],frameNum[-1]+1)
			bboxesI    = []
			for ij in range(0,4):
				interpfn  = interp1d(frameNum, bboxes[:,ij])
				bboxesI.append(interpfn(frameI))
			bboxesI  = numpy.stack(bboxesI, axis=1)
			if max(numpy.mean(bboxesI[:,2]-bboxesI[:,0]), numpy.mean(bboxesI[:,3]-bboxesI[:,1])) > minFaceSize:
				tracks.append({'frame':frameI,'bbox':bboxesI})
	return tracks

def crop_video(track, cropFile):
	vOut = cv2.VideoWriter(cropFile + 't.avi', cv2.VideoWriter_fourcc(*'XVID'), 25, (224,224))# Write video
	dets = {'x':[], 'y':[], 's':[]}
	for det in track['bbox']: # Read the tracks
		dets['s'].append(max((det[3]-det[1]), (det[2]-det[0]))/2)
		dets['y'].append((det[1]+det[3])/2) # crop center x
		dets['x'].append((det[0]+det[2])/2) # crop center y
	dets['s'] = signal.medfilt(dets['s'], kernel_size=13)  # Smooth detections
	dets['x'] = signal.medfilt(dets['x'], kernel_size=13)
	dets['y'] = signal.medfilt(dets['y'], kernel_size=13)
	for fidx, frame in enumerate(track['frame']):
		cs  = cropScale
		bs  = dets['s'][fidx]   # Detection box size
		bsi = int(bs * (1 + 2 * cs))  # Pad videos by this amount
		image = cv2.imread(flist[frame])
		frame = numpy.pad(image, ((bsi,bsi), (bsi,bsi), (0, 0)), 'constant', constant_values=(110, 110))
		my  = dets['y'][fidx] + bsi  # BBox center Y
		mx  = dets['x'][fidx] + bsi  # BBox center X
		face = frame[int(my-bs):int(my+bs*(1+2*cs)),int(mx-bs*(1+cs)):int(mx+bs*(1+cs))]
		vOut.write(cv2.resize(face, (224, 224)))
	audioTmp    = cropFile + '.wav'
	audioStart  = (track['frame'][0]) / 25
	audioEnd    = (track['frame'][-1]+1) / 25
	vOut.release()
	command = ("ffmpeg -y -i %s -async 1 -ac 1 -vn -acodec pcm_s16le -ar 16000 -threads %d -ss %.3f -to %.3f %s -loglevel panic" % \
		      (audioFilePath, nDataLoaderThread, audioStart, audioEnd, audioTmp))
	output = subprocess.call(command, shell=True, stdout=None) # Crop audio file
	command = ("ffmpeg -y -i %st.avi -i %s -threads %d -c:v copy -c:a copy %s.avi -loglevel panic" % \
			  (cropFile, audioTmp, nDataLoaderThread, cropFile)) # Combine audio and video file
	output = subprocess.call(command, shell=True, stdout=None)
	os.remove(cropFile + 't.avi')
	return {'track':track, 'proc_track':dets}

def evaluate_network(files):
	s = talkNet()
	s.loadParameters(pretrainModel)
	s.eval()
	allScores = []
	durationSet = {1,1,1,2,2,2,3,3,4,5,6} # Use this line can get more reliable result
	for file in tqdm.tqdm(files, total = len(files)):
		fileName = os.path.splitext(file.split('/')[-1])[0] # Load audio and video
		_, audio = wavfile.read(os.path.join(pycropPath, fileName + '.wav'))
		audioFeature = python_speech_features.mfcc(audio, 16000, numcep = 13, winlen = 0.025, winstep = 0.010)
		video = cv2.VideoCapture(os.path.join(pycropPath, fileName + '.avi'))
		videoFeature = []
		while video.isOpened():
			ret, frames = video.read()
			if ret == True:
				face = cv2.cvtColor(frames, cv2.COLOR_BGR2GRAY)
				face = cv2.resize(face, (224,224))
				face = face[int(112-(112/2)):int(112+(112/2)), int(112-(112/2)):int(112+(112/2))]
				videoFeature.append(face)
			else:
				break
		video.release()
		videoFeature = numpy.array(videoFeature)
		length = min((audioFeature.shape[0] - audioFeature.shape[0] % 4) / 100, videoFeature.shape[0] / 25)
		audioFeature = audioFeature[:int(round(length * 100)),:]
		videoFeature = videoFeature[:int(round(length * 25)),:,:]
		allScore = [] # Evaluation use TalkNet
		for duration in durationSet:
			batchSize = int(math.ceil(length / duration))
			scores = []
			with torch.no_grad():
				for i in range(batchSize):
					inputA = torch.FloatTensor(audioFeature[i * duration * 100:(i+1) * duration * 100,:]).unsqueeze(0).cuda()
					inputV = torch.FloatTensor(videoFeature[i * duration * 25: (i+1) * duration * 25,:,:]).unsqueeze(0).cuda()
					embedA = s.model.forward_audio_frontend(inputA)
					embedV = s.model.forward_visual_frontend(inputV)
					embedA, embedV = s.model.forward_cross_attention(embedA, embedV)
					out = s.model.forward_audio_visual_backend(embedA, embedV)
					score = s.lossAV.forward(out, labels = None)
					scores.extend(score)
			allScore.append(scores)
		allScore = numpy.round((numpy.mean(numpy.array(allScore), axis = 0)), 1).astype(float)
		allScores.append(allScore)
	return allScores

In [ ]:
# TalkNet sobre las caras de la pasada única: seguimiento por plano, recorte con audio y puntuación (> 0 = habla)
pycropPath = os.path.join(work, "crops")
shutil.rmtree(pycropPath, ignore_errors=True); os.makedirs(pycropPath)
bounds = [0] + cuts + [len(flist)]
allTracks = []
for a, b in zip(bounds[:-1], bounds[1:]):
    if b - a >= minTrack:
        allTracks.extend(track_shot([fr[:] for fr in faces[a:b]]))        # copia: track_shot vacía las listas
vidTracks = [crop_video(tr, os.path.join(pycropPath, f"{i:05d}")) for i, tr in enumerate(tqdm.tqdm(allTracks))]
scores = evaluate_network(sorted(glob.glob(f"{pycropPath}/*.avi")))

# Una fila por cara y fotograma; puntuación suavizada ±2 fotogramas como en visualization() de la demo
asd = pd.DataFrame([{"frame": fr, "x1": tr["proc_track"]["x"][i] - tr["proc_track"]["s"][i], "y1": tr["proc_track"]["y"][i] - tr["proc_track"]["s"][i],
                     "x2": tr["proc_track"]["x"][i] + tr["proc_track"]["s"][i], "y2": tr["proc_track"]["y"][i] + tr["proc_track"]["s"][i],
                     "score": float(np.mean(sc[max(i - 2, 0): min(i + 3, len(sc) - 1)]))}
                    for tr, sc in zip(vidTracks, scores) for i, fr in enumerate(tr["track"]["frame"].tolist())])
asd["talking"] = asd.score >= 0

# talking de cada persona: alguna cara que habla dentro de su caja en ese mismo fotograma
asd_by_frame = asd.groupby("frame")
def is_talking(r):
    if r.frame not in asd_by_frame.groups: return False
    c = asd_by_frame.get_group(r.frame)
    cx, cy = (c.x1 + c.x2) / 2, (c.y1 + c.y2) / 2
    return bool(c.talking[(r.x1 <= cx) & (cx <= r.x2) & (r.y1 <= cy) & (cy <= r.y2)].any())
df["talking"] = [is_talking(r) for r in df.itertuples()]
per_track["talking_s"] = df.groupby("track").talking.sum() * STRIDE / FPS

## 6. Resultado

In [ ]:
name = os.path.splitext(os.path.basename(VIDEO))[0]
per_track.round(2).to_csv(f"dataframes/tracks_{name}.tsv", sep="\t")
df.drop(columns="emb").to_csv(f"dataframes/detections_{name}.tsv", sep="\t", index=False)

personas = per_track.groupby("person").agg(gender=("gender_person", "first"), age=("age_person", "first"),
                                           screen_time_s=("screen_time_s", "sum"), talking_s=("talking_s", "sum"))
display(per_track.sort_values("start_s").round(2))
personas.sort_values("screen_time_s", ascending=False)

In [ ]:
# Vídeo anotado: persona | género | edad, en verde si está hablando
COL = {"F": (161, 86, 99), "M": (176, 143, 46), "unknown": (150, 150, 150)}   # BGR
out = os.path.join(work, "annotated.avi")
vw = cv2.VideoWriter(out, cv2.VideoWriter_fourcc(*"XVID"), FPS, cv2.imread(flist[0]).shape[1::-1])
by_frame, rows = df.groupby("frame"), []
for f, fname in enumerate(tqdm.tqdm(flist)):
    img = cv2.imread(fname)
    if f in by_frame.groups: rows = list(by_frame.get_group(f).itertuples())
    elif f % STRIDE == STRIDE - 1: rows = []                           # fotograma analizado sin personas
    for r in rows:
        g, a = per_track.gender_person[r.track], per_track.age_person[r.track]
        cv2.rectangle(img, (int(r.x1), int(r.y1)), (int(r.x2), int(r.y2)), (0, 255, 0) if r.talking else COL[g], 3)
        cv2.putText(img, f"person {r.person} | {g} | {a:.0f}", (int(r.x1) + 5, int(r.y1) + 20), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
    vw.write(img)
vw.release()
subprocess.call(f"ffmpeg -y -i {out} -i {audioFilePath} -c:v libx264 -pix_fmt yuv420p -c:a aac {work}/annotated.mp4 -loglevel panic", shell=True)
Video(f"{work}/annotated.mp4", embed=True, width=640)